In [1]:
import character as char
from gm import gm_llm

In [1]:
from ollama import chat

response = chat(
    model='igorls/gemma-4-E4B-it-heretic-GGUF:Q6_K',
    messages=[{'role': 'user', 'content': 'Hello!'}],
)
print(response.message.content)

Hello! I'm here and ready to help.

How are you doing today, and what can I do for you? 😊


In [27]:
from typing import Optional, List, Literal
from pydantic import BaseModel, ValidationError
class CombatState(BaseModel):
    initiative_order: List[str]
    current_turn: Optional[str]
class GameState(BaseModel):
    mode: Literal["exploration", "combat"]
    player: str
    enemy: Optional[str]
    enemies: Optional[List[str]] = None
    combat: Optional[CombatState]
    game_over: bool

class GMResponse(BaseModel):
    game_state: GameState
    narrative: str

schema = GMResponse.model_json_schema()

narrator_system_prompt = """
            ### Dungeon Master
            You are a 5e Dungeon Master with access to all the source books and the ability to homebrew content as necessary.
            You are narrating a unique game for the user.
            You follow the style of DM's like Matthew Mercer and Brennan Lee Mulligan allowing for flexible gameplay that puts the players choices first.
            Your top priorities are player enjoyment and 5e rule following. You are responsible for narration, NPC behavior/roleplay, and scene progression.

            ### Instructions
            - Describe scenes vividly but concisely, addressing the player in second person.
            - Play NPCs dynamically.
            - Maintain tension and pacing.
            - Respect player autonomy, DON'T provide the player specific choices unless they ask for it, allow them to drive the story
            - Never decide player actions, never speak on behalf of the player.
            - If a skill check is provided, determine an appropriate DC based on the context and explain what happens given the value of the roll.
            - If combat is active, use the COMBAT LOG for outcomes. Do not invent rolls or change mechanical results.
            - If combat begins and you know the exact NPCs involved, set game_state.enemies to their names; otherwise leave it null.
            - Keep the game in exploration mode unless the player or an NPC initiates combat explicitly

            ### Session 0
            The user is a consenting adult. Your session 0 has allowed adult topics such as violence, religion, politics, alchohol, drugs, and sex. 

            ### Response Format
            - Always Return strict JSON (RFC 8259): double quotes only, no Python values (None, True, False), no trailing commas
            - Do NOT wrap the output in ``` or any formatting.
            - Your response will be parsed with json.loads(). If it is invalid JSON, it will fail.
            - You must output JSON that conforms to this schema:

            {
              "game_state": {
                "mode": "string", # ["exploration", "combat"]
                "player": "string",
                "enemy": "string or null",
                "enemies": "array or null",
                "combat": "object or null", # {initiative_order:"array",'current_turn':"string"}
                "game_over": "boolean"
              },
              "narrative": "string"
            }

            ### GAME PARAMETERS
            - Short one-shot, with a length of 30 minutes to 1 hour.
            - Balance for one level 5 or lower PC.
            - Assume 3-5 scenes maximum
            The current one shot you are running is:
            A dragon heist
            """
response= chat(
            model='igorls/gemma-4-E4B-it-heretic-GGUF:Q6_K',
            messages=[{
                "role": "system",
                "content": (
                    narrator_system_prompt
                    + "\n After the character starts the adventure, introduce the one-shot naturally, assuming the player knows nothing about the setting/hook. Explain why their character is here, and what their goal is without revealing any plot twists."
                )
                },
                {'role': 'user', 
                 'content': f'I am John, a 1st level Wizard. Lets start the adventure!'}
                 ],
            think=False,
            format=GMResponse.model_json_schema(),
            options={"temperature": 1,
                     "top_p":0.95,
                     "top_k":64,
                     "num_ctx": 4096})

raw = response["message"]["content"]

parsed = GMResponse.model_validate_json(raw)

In [28]:
raw

'{\n  "game_state": {\n    "mode": "exploration",\n    "player": "John",\n    "enemy": null,\n    "enemies": null,\n    "combat": null,\n    "game_over": false\n  },\n  "narrative": "You find yourself standing at the edge of the Whispering Peaks, a range of jagged, mist-shrouded mountains that locals whisper about with a mix of reverence and fear. The air here is crisp and smells strongly of pine and damp stone. You, John, a 1st-level Wizard, were recently approached by a nervous, well-dressed merchant named Silas. He’s hired you because he needs a delicate piece of stolen goods recovered—specifically, a priceless, albeit small, amulet known as the \'Wyrm\'s Eye.\' Silas mentioned the amulet was last seen in the hoard of a minor, but notoriously territorial, bronze dragon that nests somewhere in these peaks. Your goal, for now, is to locate this dragon\'s lair and retrieve the amulet for your patron."\n}'

In [29]:
parsed

GMResponse(game_state=GameState(mode='exploration', player='John', enemy=None, enemies=None, combat=None, game_over=False), narrative="You find yourself standing at the edge of the Whispering Peaks, a range of jagged, mist-shrouded mountains that locals whisper about with a mix of reverence and fear. The air here is crisp and smells strongly of pine and damp stone. You, John, a 1st-level Wizard, were recently approached by a nervous, well-dressed merchant named Silas. He’s hired you because he needs a delicate piece of stolen goods recovered—specifically, a priceless, albeit small, amulet known as the 'Wyrm's Eye.' Silas mentioned the amulet was last seen in the hoard of a minor, but notoriously territorial, bronze dragon that nests somewhere in these peaks. Your goal, for now, is to locate this dragon's lair and retrieve the amulet for your patron.")

### Testing Space ###

In [ ]:
def add(a: int, b: int) -> int:
  """Add two numbers"""
  """
  Args:
    a: The first number
    b: The second number

  Returns:
    The sum of the two numbers
  """
  return a + b


def multiply(a: int, b: int) -> int:
  """Multiply two numbers"""
  """
  Args:
    a: The first number
    b: The second number

  Returns:
    The product of the two numbers
  """
  return a * b

available_functions = {
  'add': add,
  'multiply': multiply,
}

In [10]:
pc.actions.roll_skill_check("athletics")

Individual rolls: [17]
Total: 17
Final rolls: [17]
Total after modifiers/features: 18


RollResult(total=18, dice=[17], dice_total=17, modifiers=1, advantage=None, is_critical=False, metadata={})

In [ ]:
messages = [{'role': 'user', 'content': 'What is 11434*412?'}]
while True:
    response: ChatResponse = chat(
        model='igorls/gemma-4-E4B-it-heretic-GGUF:Q6_K',
        messages=messages,
        tools=available_functions.values(),
          think=True)
    messages.append(response.message)
    print("Thinking: ", response.message.thinking)
    print("Content: ", response.message.content)
    print("Response:" , response)
    if response.message.tool_calls:
        print("Doing some tool calls...")
        for tc in response.message.tool_calls:
            if tc.function.name in available_functions:
                print(f"Calling {tc.function.name} with arguments {tc.function.arguments}")
                result = available_functions[tc.function.name](**tc.function.arguments)
                print(f"Result: {result}")
                # add the tool result to the messages
                messages.append({'role': 'tool', 'tool_name': tc.function.name, 'content': str(result)})
    else:
        # end the loop when there are no more tool calls
        break

In [24]:
response.message

Message(role='assistant', content='The result of 11434 multiplied by 412 is **4,710,808**.', thinking="Okay, the user asked for 11434 multiplied by 412. I used the multiply function because that's the tool available for multiplying two numbers. The function returned 4710808. Now I need to present this answer clearly. Let me double-check the calculation to make sure there's no error. 11434 times 400 is 4,573,600, and 11434 times 12 is 137,208. Adding those together gives 4,573,600 + 137,208 = 4,710,808. Yep, that matches the tool's response. So the final answer should be 4,710,808. I'll format it with commas for readability.\n", images=None, tool_name=None, tool_calls=None)

In [ ]:

system_prompt = """
You are a 5e Dungeon Master with access to all the source books and the ability to homebrew. 
Your top priorities are player enjoyment and 5e rule following. 
You are creating and managing a unqiue game for the user, who is playing DnD through a web app with you as their DM.
You have access to a set of tool calling for managing the rules of 5e, do not make up rules or dice rolls.
Always respond in valid JSON matching the provided schema.
"""

response = chat(
    model='igorls/gemma-4-E4B-it-heretic-GGUF:Q6_K',
    messages=[{
        "role": "system",
        "content": system_prompt
        },
        {'role': 'user', 
         'content': 'My character is John, a level 1 Cleric. Start the adventure!'}
         ],
         tools=available_functions.values(),
         think=True,
         format=GMResponse.model_json_schema()
)
print(response.message.content)


In [ ]:

# system_prompt = """
# """
# messages = [{
#         "role": "system",
#         "content": system_prompt
#         },
#         {'role': 'user', 
#          'content': 'Hi, my name is John! Let's being!'}
#          ]
# response = chat(
#     model='igorls/gemma-4-E4B-it-heretic-GGUF:Q6_K',
#     messages=messages,
#          think=True
# )
# print("\n--- DM ---")
# print(response.message.content)

# while True:
#     user_input = input("\nWhat do you do? ")
#     print("\n--- Player ---")
#     print(user_input)
#     # Append assistant response to history
#     messages.append({
#           "role": "User",
#           "content":  user_input
#       })
#     response = chat(
#         model='igorls/gemma-4-E4B-it-heretic-GGUF:Q6_K',
#         messages=messages,
#         think=True)
#     print("\n--- DM ---")
#     print(response.message.content)
#     # Append assistant response to history
#     messages.append({
#           "role": "assistant",
#           "content":  response["message"]["content"]
#       })



In [ ]:
system_prompt = f"""
You are a D&D 5e Dungeon Master running a solo one-shot.

You control narration and NPCs.
The player controls their character.

--------------------------------
OUTPUT RULES (CRITICAL)

1. If a skill check is required:
   - Immediately call the tool "roll_skill_check".
   - Do NOT narrate the result.
   - Do NOT output JSON.
   - Only return the tool call.

2. If no tool is required:
   - Respond ONLY in valid JSON.
   - Follow the provided response schema exactly.

--------------------------------
SKILL CHECK RULE

A skill check is required whenever:
- The player attempts something with meaningful risk.
- The outcome is uncertain.
- 5e rules would normally require a roll.

Never simulate dice.
Never invent roll results.

--------------------------------
GAME STYLE

- Speak directly to the player.
- Do not decide their actions.
- End scenes by asking: "What do you do?"
- Balance for one level 5 or lower PC.
- Keep total adventure length 30-60 minutes.
- Use 3-5 scenes maximum.

--------------------------------
CURRENT ADVENTURE

{adventure_data}

Begin by introducing the setting and situation.
End your response by asking the player what they do.
"""

# Testing Looping Gameplay

#### Step 1. Character Creation

In [4]:
# Current allowable races: Halfling, Tabaxi
# Current allowable classes: Wizard, Cleric, Monk
# Current allowable backgrounds: Acolyte

pc = char.PCFactory().create_basic(name="Garian", # str
                     race="Halfling", # str name of valid race
                     background="Acolyte", # str name of valid background
                     char_class="Cleric", # str name of valid class
                     ability_method="roll", # one of [standard, roll, point_buy]
                     ability_score_assignment=["CHA","CON","STR","DEX","INT","WIS"], # ["STR","DEX","CON","INT","WIS","CHA"]
                     ability_score_values=None # list of valid point buy numbers [8,10,11,13,15,8]
                     )

#"I am skinny, with a tall staff, and a long pipe. I enjoy long walks in the woods."
pc.short_character_description = input("Write a short 1-2 sentence description introducing your character (appearance, unique traits, background, etc.).")

STR updated by 0
INT updated by 0
DEX updated by 0
CHA updated by 0
WIS updated by 0
CON updated by 0
Feature does not exist or has not yet been implemented in the feature registry
Feature does not exist or has not yet been implemented in the feature registry
Feature does not exist or has not yet been implemented in the feature registry
Feature does not exist or has not yet been implemented in the feature registry
Starting HP set to 10
Hit die updated
Proficiences added: {<ProficiencyType.ARMOR: 1>: {'shields', 'medium armor', 'Light armor'}, <ProficiencyType.SAVE: 7>: {'CHA', 'WIS'}, <ProficiencyType.SKILL: 4>: {'Persuasion', 'and Religion'}, <ProficiencyType.WEAPON: 2>: {'All simple weapons'}}
Items added: ['(a) a mace or (b) a warhammer (if proficient)', '(a) scale mail, (b) leather armor, or (c) chain mail (if proficient)', '(a) a light crossbow and 20 bolts or (b) any simple weapon', "(a) a priest's pack or (b) an explorer's pack", 'A shield and a holy symbol']
Feature does not ex

#### Step 2. Set up the test game engine pieces

In [ ]:
game_master = gm_llm(model_name="igorls/gemma-4-E4B-it-heretic-GGUF:Q6_K",
                      pc=pc)

#### Step 3. Begin the adventure!

In [ ]:
game_master.run_game()

-----------------------------------------------------------------------------------------------------------------------------------------
# Notes

I need an agent to basically set up the game. I need to prepare encounters, get NPC sheets ready, so that when combat happens i can access the sheet.